# 🇮🇩 Indonesian KTP Synthetic Dataset Explorer

**Dataset**: [`cloverx-id/indonesian-id-card-dummy`](https://huggingface.co/datasets/cloverx-id/indonesian-id-card-dummy)  
*(The updated Parquet mirror of `cloverxion/KTP-INDONESIA-INDONESIAN-ID-CARD-DUMMY-24K`)*

---

### 📌 How this fits into our eKYC & Anti-Spoofing Project:
1. **Zero Privacy Risk (UU PDP)**: 100% synthetic/dummy data. Safe to use and distribute.
2. **Raw Material for Presentation Attacks**:
   - **`fake_screen`**: Display these dummy cards on phone/monitor screens and photograph them.
   - **`fake_print`**: Print these cards on A4/photo paper and photograph them.
3. **Direct Training for `fake_template`**: Because these cards are digital renders, they are perfect training samples for detecting fake/generated KTPs.
4. **OCR & NIK Validator Testing**: Perfect ground-truth data to benchmark `ocr.py` and test our NIK mathematical checks.

> ⚠️ **Important Reminder**: Do **NOT** train your model on these raw images as `real_ktp`! Genuine `real_ktp` cards must be real physical cards photographed by a camera.

## 1. Environment Setup & Imports
Uncomment the pip command below if any dependencies are missing.

In [ ]:
# !pip install datasets huggingface_hub pandas matplotlib pillow pyarrow

import os
import json
import matplotlib.pyplot as plt
from PIL import Image
from datasets import load_dataset

print("✅ Setup complete. Ready to load data!")

## 2. Stream Samples on the Fly (Zero 12GB Download Needed!)

Because the full dataset is ~12-20 GB, downloading the whole dataset at once will eat up time and disk space.
We use **Hugging Face Streaming Mode** (`streaming=True`) to fetch just a few samples via HTTP stream in seconds.

In [ ]:
print("Connecting to Hugging Face dataset stream...")

# Streaming the flat card subset directly
ds = load_dataset(
    "cloverx-id/indonesian-id-card-dummy",
    data_files={"train": "data/flat/train-00000-of-00012.parquet"},
    split="train",
    streaming=True
)

# Fetch the first 6 samples
NUM_SAMPLES = 6
samples = []

for sample in ds:
    samples.append(sample)
    if len(samples) >= NUM_SAMPLES:
        break

print(f"✅ Successfully fetched {len(samples)} samples!")
print("Available fields in each record:", [k for k in samples[0].keys() if k != "image"])

## 3. Visualize KTP Cards & Field Annotations

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 14))
axes = axes.flatten()

for i, sample in enumerate(samples):
    img = sample["image"]
    nik = sample.get("nik", "N/A")
    nama = sample.get("nama", "N/A")
    ttl = sample.get("tempat_tanggal_lahir", "N/A")
    gender = sample.get("jenis_kelamin", "N/A")
    prov = sample.get("provinsi", "")
    kab = sample.get("kabupaten_kota", "")
    
    axes[i].imshow(img)
    axes[i].set_title(
        f"[{i+1}] {prov} - {kab}\nNIK: {nik} | {nama}\n{ttl} | {gender}",
        fontsize=9,
        pad=6
    )
    axes[i].axis("off")

plt.tight_layout()
plt.show()

## 4. Inspect Full Metadata (Ground Truth)
Every sample includes exact ground-truth values for all OCR fields:

In [ ]:
sample_0_meta = {k: v for k, v in samples[0].items() if k != "image"}
print("=== GROUND TRUTH JSON FOR SAMPLE #1 ===")
print(json.dumps(sample_0_meta, indent=2, ensure_ascii=False))

## 5. Test Deterministic NIK Validation (Our Checksum Engine)

Let's test our project's NIK checksum and cross-field logic against these generated records:
1. **16 Digits**: Must be exact length.
2. **Province Code**: Digits 1-2 must match official Indonesian province range (`11` to `96`).
3. **Gender Birth Day Offset**: Female day is `DD + 40` (e.g. 52 = 12th). Male is $\le 31$.

In [ ]:
def test_nik_integrity(sample):
    nik = str(sample.get("nik", "")).strip()
    gender = str(sample.get("jenis_kelamin", "")).upper()
    
    if len(nik) != 16 or not nik.isdigit():
        return f"❌ Invalid format (length {len(nik)})"
        
    prov_code = int(nik[:2])
    prov_valid = 11 <= prov_code <= 96
    
    nik_day = int(nik[6:8])
    is_female = "PEREMPUAN" in gender or "WANITA" in gender
    
    if is_female:
        gender_valid = nik_day > 40 and (nik_day - 40) <= 31
        rule_msg = f"Female offset (Day {nik_day-40} + 40 = {nik_day})"
    else:
        gender_valid = 1 <= nik_day <= 31
        rule_msg = f"Male day ({nik_day})"
        
    status = "✅ PASS" if (prov_valid and gender_valid) else "❌ FORGERY_SUSPECT"
    return f"{status} | Prov Code {prov_code} (Valid: {prov_valid}) | {rule_msg} (Valid: {gender_valid})"

print("=== BENCHMARKING NIK INTEGRITY RULES ===")
for idx, s in enumerate(samples):
    result = test_nik_integrity(s)
    print(f"Sample #{idx+1} [NIK: {s.get('nik')} | {s.get('jenis_kelamin')}]: {result}")

## 6. Export Samples to Disk
Run this cell to save the downloaded sample cards as PNG files in `dataset/dummy_samples/`.

You can now:
- Display them on your smartphone/tablet/monitor $\rightarrow$ shoot with camera $\rightarrow$ creates **`fake_screen`** data.
- Print them on an A4 sheet $\rightarrow$ shoot with camera $\rightarrow$ creates **`fake_print`** data.
- Use these clean PNGs directly as training data for the **`fake_template`** class.

In [ ]:
OUTPUT_DIR = "dummy_samples"
os.makedirs(OUTPUT_DIR, exist_ok=True)

for idx, sample in enumerate(samples):
    nik = sample.get("nik", f"sample_{idx}")
    img_path = os.path.join(OUTPUT_DIR, f"ktp_{nik}.png")
    sample["image"].save(img_path)
    print(f"Saved: {img_path}")

print(f"\n🎉 All {len(samples)} samples saved to: {OUTPUT_DIR}/")